# Calculating the Allan Variance for our dataset.

The following produces the primary unit of analysis for the current study--an calculation of allan variance that can then be used to calculate a linear value $\alpha$ for exponential decay using regression.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime as dt

import warnings; warnings.filterwarnings('ignore')

In [ ]:
DATA_LOC = '../../data'

# path to processed and ready data
DATA_PATH = os.path.join(DATA_LOC, 'lme_ready')

# path to save data to on completion
VIS_PATH = os.path.join(DATA_LOC, 'web_vis')

# post processing & ready for data visualization steps
FINAL_DOC = os.path.join(DATA_LOC, 'resids.parquet')
print(os.path.exists(FINAL_DOC))

In [ ]:
# Allan Var doc
allan_var = 'allan_var-multiling.parquet'

#### Load data

In [ ]:
# df = pd.read_table(FINAL_DOC, sep='\t')
main_df = pd.read_parquet(FINAL_DOC)
main_df.shape

In [ ]:
main_df.head()

In [ ]:
main_df['conversation_id'].nunique()

### Some specific checks

In [ ]:
main_df['null'].sum(), main_df['null'].mean()

In [ ]:
tids = main_df['groups'].loc[main_df['null']].unique()
tids

In [ ]:
main_df['groups'].loc[main_df['groups'].isin([tid.replace('null-', '') for tid in tids]) & (~main_df['null'])].nunique() / len(tids)


In [ ]:
check_df = main_df.loc[main_df['groups'].isin(tids[:3])].copy()
check_df['groups'] = [tid.replace('null-', '') for tid in check_df['groups']]

check_df = pd.merge(
    left=check_df, left_on=['groups', 'self', 'sample_delta'],
    right=main_df[['groups', 'self', 'sample_delta', 'Hxy']].loc[~main_df['null']], right_on=['groups', 'self', 'sample_delta']
)
print(list(check_df))
check_df[['groups', 'Hxy_y', 'Hxy_x']].head(n=100)

## Merge `copied doc` Method.

### Regular Data

In [ ]:
df = main_df.loc[~main_df['null']]

In [ ]:
merge_columns = ['groups', 'self', 'sample_delta']

In [ ]:
value_columns = ['Hxy']

#### First turn after the current turn

In [ ]:
# make a copy of our df
df_ = df.copy()

# for every turn_delta, subtract it by one (so 2 => 1, et al.)
df_['sample_delta'] -= 1

In [ ]:
# merge the copied doc with the actual doc using dyad ID, self, 
#   and turn_delta as indexes
df = pd.merge(
    left=df, left_on=merge_columns,
    right=df_[merge_columns+value_columns], right_on=merge_columns,
    how='left'
)

# check the shape
print(df.shape)
df.isna().sum()

#### Second turn after the current turn

In [ ]:
# for every turn_delta, subtract it by one (so 2 => 1, et al.)
df_['sample_delta'] -= 1

In [ ]:
# merge the copied doc with the actual doc using dyad ID, self, 
#   and turn_delta as indexes
df = pd.merge(
    left=df, left_on=merge_columns,
    right=df_[merge_columns+value_columns], right_on=merge_columns,
    how='left'
)

# check the shape
print(df.shape)
df.isna().sum()

#### Final checks

In [ ]:
# check what row x column combos have NaNs
df.isna().sum()

In [ ]:
# get a sample of the NaN containing rows
df[['groups', 'sample_delta', 'Hxy_x', 'Hxy_y', 'Hxy']].loc[df['Hxy'].isna()].sample(n=10).head(n=10)

In [ ]:
df = df.loc[~df['Hxy'].isna()]
df.shape

In [ ]:
df = df.rename(columns={'resid': 'residual'})
df = df.rename(columns={'Hxy_x': 'resid', 'Hxy_y': 'resid_1', 'Hxy': 'resid_2'})

In [ ]:
df[['groups', 'resid', 'resid_1', 'resid_2']].head()

#### Create Allan Variance Scale Parameter

In [ ]:
group_cols = ['groups', 'self']
number_cols = ['sample_delta']

In [ ]:
N = df[group_cols+number_cols].groupby(by=group_cols).agg('count').reset_index()
N = N.rename(columns={'sample_delta': 'N'})

In [ ]:
df = pd.merge(
    left=df, left_on=group_cols,
    right=N, right_on=group_cols,
    how='left'
)
df.shape

In [ ]:
tau = df[group_cols+number_cols].groupby(by=group_cols).agg('max').reset_index()
tau = tau.rename(columns={'sample_delta': 'tau'})

In [ ]:
df = pd.merge(
    left=df, left_on=group_cols,
    right=tau, right_on=group_cols,
    how='left'
)
df.shape

In [ ]:
df.head()

#### Calculate Allan $\sigma^2$

In [ ]:
df = df.loc[~df['resid_2'].isna()]

In [ ]:
df['allan_var'] = (1/(2*(df['tau']**2))) * ((df['resid_2'] - (2*df['resid_1']) + df['resid'])**2)

In [ ]:
df.head()

#### checkpoint

In [ ]:
df['null'] = False

In [ ]:
df.to_parquet('regular_data.parquet')

### Null data

In [ ]:
df = main_df.loc[main_df['null']]

In [ ]:
merge_columns = ['groups', 'self', 'sample_delta']

In [ ]:
value_columns = ['Hxy']

#### First turn after the current turn

In [ ]:
# make a copy of our df
df_ = df.copy()

# for every turn_delta, subtract it by one (so 2 => 1, et al.)
df_['sample_delta'] -= 1

In [ ]:
# merge the copied doc with the actual doc using dyad ID, self,
#   and turn_delta as indexes
df = pd.merge(
    left=df, left_on=merge_columns,
    right=df_[merge_columns+value_columns], right_on=merge_columns,
    how='left'
)

# check the shape
print(df.shape)
df.isna().sum()

#### Second turn after the current turn

In [ ]:
# for every turn_delta, subtract it by one (so 2 => 1, et al.)
df_['sample_delta'] -= 1

In [ ]:
# merge the copied doc with the actual doc using dyad ID, self,
#   and turn_delta as indexes
df = pd.merge(
    left=df, left_on=merge_columns,
    right=df_[merge_columns+value_columns], right_on=merge_columns,
    how='left'
)

# check the shape
print(df.shape)
df.isna().sum()

#### Final checks

In [ ]:
# check what row x column combos have NaNs
df.isna().sum()

In [ ]:
# get a sample of the NaN containing rows
df[['groups', 'sample_delta', 'Hxy_x', 'Hxy_y', 'Hxy']].loc[df['Hxy'].isna()].sample(n=10).head(n=10)

In [ ]:
df = df.loc[~df['Hxy'].isna()]
df.shape

In [ ]:
df = df.rename(columns={'resid': 'residual'})
df = df.rename(columns={'Hxy_x': 'resid', 'Hxy_y': 'resid_1', 'Hxy': 'resid_2'})

In [ ]:
df[['groups', 'resid', 'resid_1', 'resid_2']].head()

#### Create Allan Variance Scale Parameter

In [ ]:
group_cols = ['groups', 'self']
number_cols = ['sample_delta']

In [ ]:
N = df[group_cols+number_cols].groupby(by=group_cols).agg('count').reset_index()
N = N.rename(columns={'sample_delta': 'N'})

In [ ]:
df = pd.merge(
    left=df, left_on=group_cols,
    right=N, right_on=group_cols,
    how='left'
)
df.shape

In [ ]:
tau = df[group_cols+number_cols].groupby(by=group_cols).agg('max').reset_index()
tau = tau.rename(columns={'sample_delta': 'tau'})

In [ ]:
df = pd.merge(
    left=df, left_on=group_cols,
    right=tau, right_on=group_cols,
    how='left'
)
df.shape

In [ ]:
df.head()

#### Calculate Allan $\sigma^2$

In [ ]:
df = df.loc[~df['resid_2'].isna()]

In [ ]:
df['allan_var'] = (1/(2*(df['tau']**2))) * ((df['resid_2'] - (2*df['resid_1']) + df['resid'])**2)

In [ ]:
df[['groups', 'resid', 'resid_1', 'resid_2', 'allan_var']].head(n=20)

#### checkpoint

In [ ]:
df['null'] = True

In [ ]:
df.to_parquet('null_data.parquet')

# Save outputs

In [ ]:
df = [pd.read_parquet('null_data.parquet'), pd.read_parquet('regular_data.parquet')]

#### Quick check that data is actually different.

In [ ]:
df[0][['groups', 'resid', 'resid_1', 'resid_2', 'allan_var']].head()

In [ ]:
check_tids = [tid.replace('null-', '') for tid in df[0]['groups'].unique()[:10]]
df[1][['groups', 'resid', 'resid_1', 'resid_2', 'allan_var']].loc[df[1]['groups'].isin(check_tids)].head()

#### Save data to final .parquet

In [ ]:
df = pd.concat(df, ignore_index=True)

In [ ]:
# df.to_csv(
#     allan_var,
#     index=False,
#     encoding='utf-8',
#     sep='\t'
# )
df.to_parquet(allan_var)

#### Clean up data

In [ ]:
os.remove('null_data.parquet')
os.remove('regular_data.parquet')